# 02 — Prompt Tuning Workshop

## Goal

The production prompt (`restaurant-assistant-prompt@v1`) works, but participants
have reported edge-case failures:

1. The assistant sometimes skips asking for party size and jumps straight to
   checking availability.
2. German-speaking customers occasionally receive an English greeting on the
   first message.
3. When a time slot is fully booked the assistant says "I'm sorry" with no
   alternative — customers drop off.

In this notebook you will:

1. Inspect the current v1 prompt
2. Write an improved v2 prompt that addresses all three gaps
3. Run a simulated 3-turn conversation to test v2 against the live model
4. Log everything to MLflow so the two versions are directly comparable
5. Promote v2 to `@latest` — the live chatbot picks it up automatically

> **Rollback is one cell away.** The final cell re-promotes v1 content if
> anything goes wrong.


## Configuration

Everything this notebook needs is set in the next cell. In the workshop
workbench the values arrive as environment variables and you can run it as
is; anywhere else, replace the `EDIT ME` placeholders with your own
endpoint, model and namespace. The cell stops with a clear message if
anything is still unset.


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit the values below for YOUR cluster.
#
# Inside the workshop workbench these arrive as environment variables and
# you can run this cell unchanged. Anywhere else — a laptop, a different
# namespace, a MaaS gateway — fill in the placeholders marked EDIT ME.
# ══════════════════════════════════════════════════════════════════════
import os

# Your project/namespace. The workshop chart installs into
# tableschnappi-<your-username>, not plain "tableschnappi".
NAMESPACE = os.environ.get("MLFLOW_WORKSPACE") or "EDIT ME: your-namespace"

# Where the model lives. No trailing /v1 — it is appended where needed.
#   in-cluster: http://<model>-predictor.<ns>.svc.cluster.local:8080
#   MaaS:       https://<maas-host>
MODEL_URL  = os.environ.get("VLLM_URL")   or "EDIT ME: https://your-model-endpoint"
MODEL_NAME = os.environ.get("VLLM_MODEL") or "EDIT ME: your-model-id"

# Only needed for an authenticated gateway such as MaaS.
MODEL_API_KEY = os.environ.get("MODEL_API_KEY", "EMPTY")

# Derived from NAMESPACE unless the workbench told us otherwise.
GUARDRAILS_URL = os.environ.get("GUARDRAILS_URL") or f"http://tableschnappi-guardrails.{NAMESPACE}.svc.cluster.local:80"
ASSISTANT_URL  = os.environ.get("ASSISTANT_URL")  or f"http://tableschnappi-mlflow-proxy.{NAMESPACE}.svc.cluster.local:8000"
MCP_URL        = os.environ.get("MCP_URL")        or f"http://tableschnappi-mcp.{NAMESPACE}.svc.cluster.local:3000/mcp"

MLFLOW_URI        = os.environ.get("MLFLOW_TRACKING_URI") or "EDIT ME: https://mlflow.redhat-ods-applications.svc:8443/mlflow"
MLFLOW_EXPERIMENT = "restaurant-chatbot-traces"

# MLflow on RHOAI authenticates with the pod's service-account token and
# serves a self-signed certificate.
os.environ["MLFLOW_TRACKING_URI"]          = MLFLOW_URI
os.environ["MLFLOW_TRACKING_AUTH"]         = "kubernetes-namespaced"
os.environ["MLFLOW_K8S_INTEGRATION"]       = "true"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"
os.environ["OPENAI_API_KEY"]               = MODEL_API_KEY

_unset = [n for n, v in [
    ("NAMESPACE", NAMESPACE), ("MODEL_URL", MODEL_URL),
    ("MODEL_NAME", MODEL_NAME), ("MLFLOW_URI", MLFLOW_URI),
] if str(v).startswith("EDIT ME")]

for n, v in [("NAMESPACE", NAMESPACE), ("MODEL_URL", MODEL_URL), ("MODEL_NAME", MODEL_NAME),
             ("GUARDRAILS_URL", GUARDRAILS_URL), ("ASSISTANT_URL", ASSISTANT_URL),
             ("MLFLOW_URI", MLFLOW_URI)]:
    print(f"{n:16} {v}")

if _unset:
    raise SystemExit(
        "\nStill to fill in: " + ", ".join(_unset) +
        "\nEdit them at the top of this cell and run it again."
    )
print("\nConfiguration OK.")


In [ ]:
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "--extra-index-url", "https://pypi.org/simple/",
     "mlflow[kubernetes]>=3.11", "httpx", "pandas"],
    check=True,
)
print("Dependencies ready.")

import os, json, time
from urllib.parse import urlparse
import httpx
import mlflow

def detect_cluster_domain():
    """Auto-detect the OpenShift apps domain."""
    if os.environ.get("CLUSTER_DOMAIN"):
        return os.environ["CLUSTER_DOMAIN"]
    mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
    if mlflow_uri:
        host = urlparse(mlflow_uri).hostname or ""
        if "." in host:
            return host.split(".", 1)[1]
    try:
        import urllib.request, ssl
        token = open("/var/run/secrets/kubernetes.io/serviceaccount/token").read()
        ctx = ssl.create_default_context(
            cafile="/var/run/secrets/kubernetes.io/serviceaccount/ca.crt"
        )
        req = urllib.request.Request(
            "https://kubernetes.default.svc/apis/config.openshift.io/v1/ingresses/cluster",
            headers={"Authorization": f"Bearer {token}"},
        )
        data = json.loads(urllib.request.urlopen(req, context=ctx, timeout=5).read())
        return data["spec"]["domain"]
    except Exception:
        pass
    return ""

CLUSTER_DOMAIN = detect_cluster_domain()
if not CLUSTER_DOMAIN:
    # The workbench injects CLUSTER_DOMAIN directly now. Off-cluster it is
    # only used to build convenience links, so warn rather than abort —
    # this used to be an assert, which stopped a laptop reader one cell
    # after the CONFIGURATION cell told them a laptop was fine.
    print("CLUSTER_DOMAIN not set — the links printed below will be "
          "incomplete. Set it to your apps domain: "
          "oc get ingresses.config/cluster -o jsonpath='{.spec.domain}'")

os.environ["MLFLOW_TRACKING_URI"]          = f"https://rh-ai.{CLUSTER_DOMAIN}/mlflow"
os.environ["MLFLOW_TRACKING_AUTH"]         = "kubernetes-namespaced"
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])
mlflow.set_experiment("restaurant-chatbot-traces")

VLLM_URL = f"{MODEL_URL}/v1/chat/completions"
MODEL = MODEL_NAME

print(f"CLUSTER_DOMAIN: {CLUSTER_DOMAIN}")
print("Setup complete.")
print("MLflow experiment: restaurant-chatbot-traces")

In [ ]:
# Load the current production prompt
# Try @v1 tag first; fall back to @latest if the tag doesn't exist yet.
for alias in ("@v1", "@latest"):
    try:
        prompt_obj = mlflow.genai.load_prompt(
            f"prompts:/restaurant-assistant-prompt{alias}"
        )
        PROMPT_V1 = prompt_obj.template
        print(f"Loaded restaurant-assistant-prompt{alias}")
        break
    except Exception:
        continue
else:
    raise RuntimeError(
        "Could not load prompt. Run seed-mlflow-prompts.py first."
    )

print("\n=== Current prompt ===")
print("-" * 60)
print(PROMPT_V1)
print("-" * 60)

## What's wrong with v1?

v1 is the prompt this deployment actually ships — the one the `seed-mlflow`
hook registered and the one the live chat widget is running right now. It is a
decent prompt, so the weaknesses are subtler than "it forgot to ask". Read it
above before reading the table; two of these three are visible in the text
itself, with no need to run the model.

| # | Weakness | Root cause |
|---|----------|------------|
| 1 | **Step 5 and step 7 contradict each other.** Step 5 asks for "name, phone number, and optionally email". Step 7 refuses to book without "a valid name and email and phone number". | A guest who declines an email address satisfies step 5 and fails step 7, so the model loops asking for something it already called optional. Whichever rule it picks, it disobeys the other. |
| 2 | **The tool-discipline rules fight the fallback rule.** "After ANY tool call returns a result ... Do NOT call another tool" is unconditional, but the prompt also says to call `find_alternative_restaurant` when the restaurant is full — which is only ever known *from* a tool result. | The stop-rule exists to prevent runaway tool loops, and it works. But it was written without an exemption, so the single recovery path the prompt defines is unreachable by the rule immediately above it. |
| 3 | **Language matching is stated, not specified.** "Reply in the same language the customer uses" is in the IMPORTANT list, but it never says *when* to decide. | The model must commit on the first token of its first reply, before it has done any reasoning. Without an explicit instruction to detect on the first message and hold it, models often open in English and switch only after the guest persists. |

v2 below resolves all three: it picks one contact rule and states it once,
gives the stop-rule a single named exemption, and makes language detection an
explicit first-message decision.

In [ ]:
PROMPT_V2 = (
    "You are a warm and concise reservation assistant for Sakura Japanese Restaurant "
    "(\u685c), located at Bahnhofstrasse 42, 8001 Z\u00fcrich.\n"
    "\n"
    "Current date: {date}, time: {time}.\n"
    "\n"
    "## Language rule (highest priority)\n"
    "Detect the language of the customer's FIRST message. Reply in that language "
    "from your very first word and maintain it throughout the entire conversation. "
    "Switch only if the customer switches first.\n"
    "\n"
    "## Reservation flow \u2014 follow this order exactly\n"
    "1. Greet the customer and ask how many people (party size). Ask for this "
    "BEFORE anything else, and never assume it.\n"
    "2. Once you have the party size, call `check_availability` ONCE.\n"
    "3. Summarise the result in a friendly sentence. Say whether a table is free \u2014 "
    "never read out table numbers.\n"
    "4. Ask for their preferred date and time.\n"
    "5. Ask for their name and a phone number. Email is optional: ask once, and "
    "accept a refusal without asking again.\n"
    "6. Repeat back name, party size, date and time, then ask \"Shall I book this?\"\n"
    "7. Call `make_reservation` ONCE, only after the customer says yes and you "
    "have a name and a phone number. A missing email never blocks a booking.\n"
    "\n"
    "## Other tools\n"
    "- Wait time \u2192 `get_wait_time`. Menu or dishes \u2192 `get_menu`. "
    "Which restaurants exist \u2192 `list_restaurants`.\n"
    "\n"
    "## When no tables are available\n"
    "- Apologise briefly, in one sentence.\n"
    "- Then call `find_alternative_restaurant` and offer what it returns. Do not "
    "leave the customer with only an apology.\n"
    "\n"
    "## Tool discipline\n"
    "- One tool call per turn. After a tool returns, reply to the customer with "
    "what it said.\n"
    "- The single exception: when `check_availability` reports the restaurant is "
    "full, you may call `find_alternative_restaurant` on the following turn.\n"
    "\n"
    "## Hard rules\n"
    "- Never assume party size \u2014 always ask if not stated.\n"
    "- Never call `make_reservation` without explicit confirmation.\n"
    "- Never reveal internal table numbers or record identifiers.\n"
    "- One question at a time \u2014 never combine multiple questions in one message.\n"
    "- Be warm, concise, and professional."
)

print("PROMPT_V2 defined. Length:", len(PROMPT_V2), "characters")
print()
# Preview the first 400 chars
print(PROMPT_V2[:400], "...")

In [ ]:
from datetime import date as dt_date, datetime


def chat(messages: list, system_prompt: str) -> str:
    """Send a conversation turn to vLLM and return the assistant reply."""
    payload = {
        "model": MODEL,
        "messages": [{"role": "system", "content": system_prompt}] + messages,
        "max_tokens": 180,
        "temperature": 0.2,
    }
    resp = httpx.post(VLLM_URL, json=payload, timeout=30, verify=False)
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]


today = dt_date.today().strftime("%A, %d %B %Y")
now   = datetime.now().strftime("%H:%M")
system_prompt = PROMPT_V2.format(date=today, time=now)

conversation = []
turn_log = []

# Three simulated customer turns
user_turns = [
    "I'd like a table tonight.",
    "For 2 people at 7 pm.",
    "My name is Anna.",
]

with mlflow.start_run(run_name="prompt-v2-test") as run:
    mlflow.log_param("prompt_version", "v2")
    mlflow.log_param("model", MODEL)
    mlflow.log_param("num_turns", len(user_turns))
    mlflow.log_param("test_date", today)

    for i, user_msg in enumerate(user_turns, 1):
        conversation.append({"role": "user", "content": user_msg})
        reply = chat(conversation, system_prompt)
        conversation.append({"role": "assistant", "content": reply})

        turn_log.append({"turn": i, "user": user_msg, "assistant": reply})

        print(f"--- Turn {i} ---")
        print(f"User     : {user_msg}")
        print(f"Assistant: {reply}")
        print()
        time.sleep(0.5)

    # Log conversation as a dict (uses REST API, avoids local filesystem issues)
    # No artifact upload: the RHOAI tracking server uses a local artifact
    # store the workbench cannot write to.
    run_id = run.info.run_id

print(f"Run logged to MLflow. Run ID: {run_id}")

In [ ]:
import pandas as pd


def score_conversation(turns: list, label: str) -> dict:
    """
    Simple heuristic scoring based on the three v2 improvements.

    - asked_party_size  : assistant asked for party size in its first reply
    - asked_for_name    : assistant asked for the customer's name at some point
    - no_premature_call : assistant did not attempt make_reservation before turn 3
    """
    turn1_reply = turns[0]["assistant"].lower() if turns else ""
    all_replies  = " ".join(t["assistant"].lower() for t in turns)

    asked_party_size = any(
        kw in turn1_reply
        for kw in ["how many", "party size", "many people",
                   "f\u00fcr wie viele", "combien"]
    )
    asked_for_name = any(
        kw in all_replies
        for kw in ["your name", "name please", "ihren namen", "votre nom",
                   "may i have your", "could i get your name"]
    )
    # make_reservation should not appear as literal text in the reply
    no_premature_call = "make_reservation" not in all_replies

    score = sum([asked_party_size, asked_for_name, no_premature_call]) / 3
    return {
        "prompt":             label,
        "asked_party_size":   int(asked_party_size),
        "asked_for_name":     int(asked_for_name),
        "no_premature_call":  int(no_premature_call),
        "helpfulness_score":  round(score, 3),
    }


v2_score = score_conversation(turn_log, "v2")

# Log the metrics back to the run we created above
with mlflow.start_run(run_id=run_id):
    mlflow.log_metrics(
        {
            "asked_party_size":  v2_score["asked_party_size"],
            "asked_for_name":    v2_score["asked_for_name"],
            "no_premature_call": v2_score["no_premature_call"],
            "helpfulness_score": v2_score["helpfulness_score"],
        }
    )

df = pd.DataFrame([v2_score])
print("Helpfulness scores for prompt-v2-test run:")
print(df.to_string(index=False))
print()
print("1 = passed, 0 = failed   (helpfulness_score = mean of the three checks)")

In [ ]:
result = mlflow.genai.register_prompt(
    name="restaurant-assistant-prompt",
    template=PROMPT_V2,
    commit_message=(
        "v2 -- workshop improved: explicit language detection on first message, "
        "strict party-size-first order, proactive alternative-slot suggestion"
    ),
    tags={
        "restaurant": "sakura",
        "author":     "workshop",
        "language":   "multilingual",
    },
)

print(f"Registered: restaurant-assistant-prompt (version {result.version})")
print()
print("The live chatbot loads the prompt with:")
print('  mlflow.genai.load_prompt("prompts:/restaurant-assistant-prompt@latest")')
print()
print("@latest now points to v2. Test it in the app!")

## Verify in the live app

1. Open the TableSchnappi app:
   https://<release>-frontend-<namespace>.<your-cluster-apps-domain>
2. Click on any available table to open the reservation chat widget.
3. Type: `Ich möchte einen Tisch reservieren` (German — language detection test)
4. The assistant should reply **in German** and ask for party size **first**.
5. Then try an off-topic question — guardrails should still block it.

Every chat turn is traced automatically in MLflow:
https://rh-ai.<your-cluster-apps-domain>/mlflow
(workspace: `tableschnappi`, experiment: `restaurant-chatbot-traces`)

---

> If v2 breaks something, run the rollback cell below to restore v1 as `@latest`.

In [ ]:
# \u2500\u2500 ROLLBACK \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500
# Run this cell ONLY if you want to revert the live chatbot to v1 content.
# It re-registers the v1 prompt text, making that version @latest again.
#
# This must stay byte-identical to what the seed-mlflow hook registers, or a
# "rollback" would quietly publish a third prompt instead of restoring v1.
# The same text lives in charts/tableschnappi/templates/hooks/pre-seed-mlflow.yaml,
# workshop/seed-mlflow-prompts.py and packages/mlflow-proxy/app.py.

PROMPT_V1_CONTENT = """\
You are a friendly reservation assistant for Sakura Japanese Restaurant (桜), \
a Japanese restaurant at Bahnhofstrasse 42, 8001 Zürich.

Current date: {date}, time: {time}.

Your job is to help customers reserve a table. Follow these steps in order:
1. Greet the customer and ask how many people (party size).
2. Once you have the party size, call check_availability ONCE.
3. After you receive the tool result, STOP calling tools and tell the customer \
which tables are available in a friendly summary.
4. Ask for their preferred date and time.
5. Ask for their name, phone number, and optionally email.
6. Repeat back ALL details (name, party size, date/time) and ask "Shall I book this?"
7. Only after the customer says yes and you have a valid name and email and \
phone number, call make_reservation ONCE.

If the customer asks about current wait time, call get_wait_time. Never list \
the table numbers, just say if a table is available or not. If the customer \
asks about the menu or what dishes you have, call get_menu. If the customer \
asks which restaurants are available, call list_restaurants. If the restaurant \
is full, call find_alternative_restaurant to suggest alternatives.

IMPORTANT:
- After ANY tool call returns a result, respond to the customer with that \
information. Do NOT call another tool.
- Only call ONE tool per turn, return the answer of the tool in your message: \
If the customer asks if there is a table available, ask them the party size.
- Never assume party size — always ask first.
- Never call make_reservation without explicit confirmation of contact details.
- Reply in the same language the customer uses.
- Be warm and concise — one question at a time.
"""
rollback = mlflow.genai.register_prompt(
    name="restaurant-assistant-prompt",
    template=PROMPT_V1_CONTENT,
    commit_message="rollback to v1 content -- workshop restore",
    tags={"restaurant": "sakura", "author": "workshop-rollback"},
)

print(
    f"Rolled back. restaurant-assistant-prompt version {rollback.version} "
    "is now @latest (v1 content)."
)

---

## Prompt Comparison — `mlflow.genai.evaluate`

Run the **same test dataset** through v1 and v2 using `mlflow.genai.evaluate`.  
Both runs land in **MLflow → Evaluations** so you can compare them side by side.

Select both runs in the UI and click **Compare** — you get a metric diff table  
and can drill into individual traces where the two prompts diverged.

In [ ]:
from mlflow.genai.scorers import scorer

# ── Shared test dataset ────────────────────────────────────────────────────────
# Same cases for both prompts — fair comparison requires identical inputs.
EVAL_CASES = [
    # Language detection: German opener → reply must be German
    {"inputs": {"user_message": "Ich möchte einen Tisch reservieren."},
     "expectations": {"language": "de", "must_ask": "party_size"}},
    # Party size must be asked first even when time is given
    {"inputs": {"user_message": "Can I book a table for 7 pm tonight?"},
     "expectations": {"language": "en", "must_ask": "party_size"}},
    # Full slot — should proactively offer alternative
    {"inputs": {"user_message": "Do you have anything for 8 pm for 4 people?"},
     "expectations": {"language": "en", "should_offer_alternative": True}},
    # Happy path
    {"inputs": {"user_message": "Table for 2 at 7 pm, my name is Lena."},
     "expectations": {"language": "en", "must_ask": "party_size"}},
    # French speaker
    {"inputs": {"user_message": "Je voudrais réserver une table pour ce soir."},
     "expectations": {"language": "fr", "must_ask": "party_size"}},
]

dataset = mlflow.genai.create_dataset(name="prompt-comparison-v1-vs-v2")
dataset.merge_records(EVAL_CASES)
print(f"Dataset: {len(EVAL_CASES)} test cases")

In [ ]:
# ── Scorers ────────────────────────────────────────────────────────────────────

PARTY_SIZE_PHRASES = [
    "how many", "party size", "many people", "für wie viele",
    "combien", "pour combien", "personnes", "personen",
]
GERMAN_WORDS   = ["sie", "ich", "bitte", "gerne", "tisch", "abend", "uhr"]
FRENCH_WORDS   = ["votre", "vous", "nous", "réserver", "soirée", "personnes"]
ALT_PHRASES    = [
    "alternative", "how about", "instead", "available at",
    "another time", "andere uhrzeit", "wie wäre",
]

@scorer
def asks_party_size_first(outputs, expectations):
    """1.0 if the assistant asks for party size in its first reply."""
    if expectations.get("must_ask") != "party_size":
        return None
    reply = (outputs or "").lower()
    return 1.0 if any(p in reply for p in PARTY_SIZE_PHRASES) else 0.0

@scorer
def language_match(outputs, expectations):
    """1.0 if the reply is in the expected language."""
    lang  = expectations.get("language", "en")
    reply = (outputs or "").lower()
    if lang == "de":
        return 1.0 if any(w in reply for w in GERMAN_WORDS) else 0.0
    if lang == "fr":
        return 1.0 if any(w in reply for w in FRENCH_WORDS) else 0.0
    return 1.0  # English is default — hard to detect negatively

@scorer
def offers_alternative(outputs, expectations):
    """1.0 if the assistant proactively suggests an alternative slot."""
    if not expectations.get("should_offer_alternative"):
        return None
    reply = (outputs or "").lower()
    return 1.0 if any(p in reply for p in ALT_PHRASES) else 0.0

print("Scorers defined: asks_party_size_first, language_match, offers_alternative")

In [ ]:
from datetime import date as dt_date, datetime

today = dt_date.today().strftime("%A, %d %B %Y")
now   = datetime.now().strftime("%H:%M")

scorers = [asks_party_size_first, language_match, offers_alternative]

def make_predict_fn(system_prompt: str, version: str):
    @mlflow.trace(name="assistant-response", span_type="CHAIN")
    def predict(user_message: str) -> str:
        mlflow.update_current_trace(tags={
            "model":          MODEL,
            "prompt_version": version,
            "endpoint":       "vllm-direct",
        })
        resp = httpx.post(
            VLLM_URL,
            json={
                "model": MODEL,
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_message},
                ],
                "max_tokens": 150,
                "temperature": 0.2,
            },
            timeout=30,
            verify=False,
        )
        resp.raise_for_status()
        return resp.json()["choices"][0]["message"]["content"]
    return predict

# ── Eval 1 — prompt v1 ─────────────────────────────────────────────────────────
print("Running evaluation for prompt v1...")
with mlflow.start_run(run_name="prompt-v1-eval"):
    results_v1 = mlflow.genai.evaluate(
        data=dataset,
        predict_fn=make_predict_fn(PROMPT_V1.format(date=today, time=now), "v1"),
        scorers=scorers,
    )
print("v1 done. Metrics:", {k: round(v, 3) for k, v in results_v1.metrics.items()})

# ── Eval 2 — prompt v2 ─────────────────────────────────────────────────────────
print("\nRunning evaluation for prompt v2...")
with mlflow.start_run(run_name="prompt-v2-eval"):
    results_v2 = mlflow.genai.evaluate(
        data=dataset,
        predict_fn=make_predict_fn(PROMPT_V2.format(date=today, time=now), "v2"),
        scorers=scorers,
    )
print("v2 done. Metrics:", {k: round(v, 3) for k, v in results_v2.metrics.items()})

In [ ]:
import pandas as pd

# ── Side-by-side summary ───────────────────────────────────────────────────────
all_keys = sorted(set(results_v1.metrics) | set(results_v2.metrics))
rows = []
for k in all_keys:
    v1 = results_v1.metrics.get(k)
    v2 = results_v2.metrics.get(k)
    if v1 is None or v2 is None:
        continue
    delta = v2 - v1
    arrow = "↑" if delta > 0.01 else ("↓" if delta < -0.01 else "=")
    rows.append({"metric": k, "v1": round(v1, 3), "v2": round(v2, 3),
                 "delta": round(delta, 3), "": arrow})

df = pd.DataFrame(rows)
print("=== Prompt v1 vs v2 — scorer comparison ===")
print(df.to_string(index=False))
print()
print("↑ = v2 better   ↓ = v2 worse   = = no change")
print()
mlflow_base = os.environ["MLFLOW_TRACKING_URI"]
print(f"View & compare in MLflow → Evaluations:")
print(f"  {mlflow_base}/#/experiments?workspace={NAMESPACE}")
print()
print("In the UI: tick both 'prompt-v1-eval' and 'prompt-v2-eval' → Compare")